# 🔹 Step 0 — Install Required Libraries

In [ ]:
# Core ML + data
!pip install pandas numpy scikit-learn

# LangChain & RAG tools
!pip install langchain langchain-community openai faiss-cpu

# Multi-agent orchestration
!pip install crewai

# Optional: for visualization / demo
!pip install matplotlib seaborn streamlit

# Install streamlit if not installed
!pip install streamlit

# Run the app
!streamlit run streamlit_app.py


***Autonomous Multi-Agent Intrusion Detection System***

✔ Data preprocessing
✔ Feature engineering
✔ Label encoding
✔ Balanced dataset
✔ Isolation Forest anomaly detection
✔ RandomForest threat classification
✔ Multi-agent architecture
✔ Orchestrator agent
✔ Alert agent
✔ Live simulation
✔ Streamlit dashboard

🔹 STEP 0 — Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np
import glob
import os

# ML
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest, RandomForestClassifier
import joblib

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


# 🔹 Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 🔹 Step 2 — Check Dataset Path

In [ ]:
import os
import glob
import pandas as pd

# 1️⃣ Your confirmed folder path
Dataset_root = "path_to_dataset"

# 2️⃣ Find all CSVs recursively
csv_files = glob.glob(os.path.join(Dataset_root, "**/*.csv"), recursive=True)
print(f"Found {len(csv_files)} CSV files:")
for f in csv_files[:10]:  # show first 10 files
    print(f)

# 3️⃣ Load all CSVs into a single DataFrame
df_list = []
for file in csv_files:
    print(f"Loading {file} ...")
    df = pd.read_csv(file, low_memory=False)
    df_list.append(df)

data = pd.concat(df_list, ignore_index=True)
print("✅ Combined dataset shape:", data.shape)


# 🔹 Step 3 — Clean the Dataset

In [ ]:
# Strip extra spaces from column names
data.columns = data.columns.str.strip()

# Strip spaces from Label column
data["Label"] = data["Label"].str.strip()

# Replace infinite values with NaN
data.replace([np.inf, -np.inf], np.nan, inplace=True)

# Check missing values
print("Missing values per column:")
print(data.isna().sum().sort_values(ascending=False).head())

# Fill missing numeric values safely
numeric_cols = data.select_dtypes(include=[np.number]).columns
data[numeric_cols] = data[numeric_cols].fillna(0)

print("✅ Shape after cleaning:", data.shape)

# 🔹 Step 4 — Filter Relevant Attack Types

In [ ]:
# Define label patterns (safer for CIC-IDS dataset)
attack_patterns = ["BENIGN", "DDoS", "PortScan", "Brute Force"]

# Keep rows that contain any of these keywords
data = data[
    data["Label"].str.contains("|".join(attack_patterns), regex=True)
]

print("Label distribution after filtering:")
print(data["Label"].value_counts())

# 🔹 Step 5 — Select Features

In [ ]:

selected_features = [
    "Destination Port",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Mean",
    "Flow Bytes/s",
    "Flow Packets/s"
]

# Ensure features exist
missing_features = [f for f in selected_features if f not in data.columns]
if missing_features:
    raise ValueError(f"Missing features: {missing_features}")

X = data[selected_features]
y = data["Label"]

print("✅ Feature selection complete.")
print("Feature shape:", X.shape)

# 🔹 Step 6 — Encode Labels

In [ ]:
from sklearn.preprocessing import LabelEncoder
import joblib
import os

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

os.makedirs("models", exist_ok=True)
joblib.dump(encoder, "models/label_encoder.pkl")

print("Label Mapping:")
print(dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))

Label Mapping:
{'BENIGN': np.int64(0), 'DDoS': np.int64(1), 'PortScan': np.int64(2), 'Web Attack � Brute Force': np.int64(3)}


# 🔹 Step 7 — Balance Dataset (Optional)

In [ ]:
balanced_data = (
    data.groupby("Label", group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 5000), random_state=42))
)

X_bal = balanced_data[selected_features]
y_bal = encoder.transform(balanced_data["Label"])

print("Balanced counts:")
print(balanced_data["Label"].value_counts())

# 🔹 Step 8 — Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal,
    test_size=0.2,
    random_state=42,
    stratify=y_bal
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# 🔹 Step 9 — Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "models/scaler.pkl")

print("✅ Scaling complete.")

***🔹 ✅ Dataset is Now Ready***

X_train, X_test, y_train, y_test → ready for modeling

Data is cleaned, filtered, features selected, labels encoded, balanced

# 🔹 Full Agentic AI System

In [ ]:
# ============================================
# 🔹 STEP 10 — Train Anomaly Detector
# ============================================

from sklearn.ensemble import IsolationForest

# Identify BENIGN class index
benign_index = encoder.transform(["BENIGN"])[0]

# Train only on benign flows
X_train_benign = X_train_scaled[y_train == benign_index]

anomaly_model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)

anomaly_model.fit(X_train_benign)

joblib.dump(anomaly_model, "models/anomaly_detector.pkl")

print("✅ Anomaly model trained on BENIGN traffic only.")

✅ Anomaly model trained on BENIGN traffic only.


In [ ]:
# ============================================
# 🔹 STEP 11 — Train Threat Classifier
# ============================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

threat_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

threat_model.fit(X_train_scaled, y_train)

y_pred = threat_model.predict(X_test_scaled)

print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

joblib.dump(threat_model, "models/threat_classifier.pkl")

print("✅ Threat model saved.")

In [ ]:
# ============================================
# 🔹 STEP 12 — Multi-Agent System
# ============================================

class AnomalyAgent:
    def __init__(self, model):
        self.model = model

    def detect(self, X_scaled):
        preds = self.model.predict(X_scaled)
        return np.where(preds == -1, "Anomaly", "Normal")


class ThreatAgent:
    def __init__(self, model, encoder):
        self.model = model
        self.encoder = encoder

    def classify(self, X_scaled):
        preds = self.model.predict(X_scaled)
        return self.encoder.inverse_transform(preds)


class AlertAgent:
    def generate(self, report):
        alerts = []
        for _, row in report.iterrows():
            if row["AnomalyStatus"] == "Anomaly":
                alerts.append("⚠️ Investigate immediately")
            elif row["ThreatPrediction"] != "BENIGN":
                alerts.append(f"⚠️ {row['ThreatPrediction']} attack detected")
            else:
                alerts.append("✅ Normal traffic")
        report["AlertMessage"] = alerts
        return report


class PlannerAgent:
    def __init__(self, anomaly_agent, threat_agent, alert_agent, scaler):
        self.anomaly_agent = anomaly_agent
        self.threat_agent = threat_agent
        self.alert_agent = alert_agent
        self.scaler = scaler

    def analyze(self, new_data):
        new_data = new_data[selected_features]
        X_scaled = self.scaler.transform(new_data)

        anomaly_status = self.anomaly_agent.detect(X_scaled)
        threat_pred = self.threat_agent.classify(X_scaled)

        report = pd.DataFrame({
            "AnomalyStatus": anomaly_status,
            "ThreatPrediction": threat_pred
        })

        return self.alert_agent.generate(report)

# 🔥 STEP 13 — Initialize Agents

In [ ]:
anomaly_loaded = joblib.load("models/anomaly_detector.pkl")
threat_loaded = joblib.load("models/threat_classifier.pkl")
scaler_loaded = joblib.load("models/scaler.pkl")
encoder_loaded = joblib.load("models/label_encoder.pkl")

anomaly_agent = AnomalyAgent(anomaly_loaded)
threat_agent = ThreatAgent(threat_loaded, encoder_loaded)
alert_agent = AlertAgent()

planner = PlannerAgent(
    anomaly_agent,
    threat_agent,
    alert_agent,
    scaler_loaded
)


# 🔥 STEP 14 — Prediction Function

In [ ]:
def predict_network_flow(new_flows: pd.DataFrame):
    return planner.analyze(new_flows)

# 🔥 STEP 15 — Example Test

In [ ]:
sample_flows = X_test.iloc[:5]
print(predict_network_flow(sample_flows))

***🔹 How This Works***

AnomalyDetectionAgent → detects unusual network flows using Isolation Forest

ThreatClassificationAgent → predicts exact attack type (DDoS, PortScan, Brute Force, BENIGN)

OrchestratorAgent → combines results, outputs a report

Models & scaler are saved in models/ folder for later use.

# 🔹 STEP 16 — Make a Reusable Preprocessing & Prediction

In [ ]:
# Step 16 — CyberSecurityIDS class "The BRAIN of the System"
import joblib
import pandas as pd
import numpy as np

class CyberSecurityIDS:
    def __init__(self, models_path="models/"):
        self.selected_features = [
            "Destination Port", "Flow Duration", "Total Fwd Packets",
            "Total Backward Packets", "Total Length of Fwd Packets",
            "Total Length of Bwd Packets", "Fwd Packet Length Mean",
            "Bwd Packet Length Mean", "Flow Bytes/s", "Flow Packets/s"
        ]
        self.scaler = joblib.load(f"{models_path}/scaler.pkl")
        self.anomaly_model = joblib.load(f"{models_path}/anomaly_detector.pkl")
        self.threat_model = joblib.load(f"{models_path}/threat_classifier.pkl")
        self.encoder = joblib.load(f"{models_path}/label_encoder.pkl")

    def preprocess(self, df):
        df_clean = df[self.selected_features].copy()
        return self.scaler.transform(df_clean)

    def predict(self, df):
        X_scaled = self.preprocess(df)
        anomaly = np.where(self.anomaly_model.predict(X_scaled)==-1, "Anomaly", "Normal")
        threat = self.encoder.inverse_transform(self.threat_model.predict(X_scaled))

        alerts = []
        for a, t in zip(anomaly, threat):
            if a=="Anomaly":
                alerts.append(f"⚠️ ALERT: Anomaly detected! Threat may be {t}")
            elif t!="BENIGN":
                alerts.append(f"⚠️ ALERT: {t} attack predicted!")
            else:
                alerts.append("✅ Normal")

        report = df.copy()
        report["AnomalyStatus"] = anomaly
        report["ThreatPrediction"] = threat
        report["AlertMessage"] = alerts
        return report

# 🔹 STEP 17 — Add Visualization & Explainability (SHAP)

In [ ]:
import shap

# Use a subset to save memory
X_sample = X_train_scaled[:500]

explainer = shap.TreeExplainer(threat_model)
shap_values = explainer.shap_values(X_sample)

# Summary plot
shap.summary_plot(shap_values, X_sample, feature_names=selected_features)

# 🔹 STEP 18 — Real-Time IDS Dashboard (Streamlit)

In [ ]:
import streamlit as st
import pandas as pd
import time

# Initialize IDS system (Step 16 class must be defined already)
ids_system = CyberSecurityIDS(models_path="models/")

st.title("🚨 Live Intrusion Detection Dashboard")
uploaded_file = st.file_uploader("Upload CSV", type=["csv"])

# Only proceed if file is uploaded
if uploaded_file is not None:
    df_flows = pd.read_csv(uploaded_file)
    st.subheader("📊 Data Preview")
    st.dataframe(df_flows.head())

    # Simulation placeholders
    table_placeholder = st.empty()
    anomaly_chart_placeholder = st.empty()
    attack_chart_placeholder = st.empty()
    metrics_placeholder = st.empty()

    # Start simulation
    st.write("🚀 Starting live simulation...")
    flow_idx = 0
    num_flows = len(df_flows)
    anomaly_counts = []
    attack_counts = []

    while True:
        new_flow = df_flows.iloc[[flow_idx % num_flows]]
        flow_idx += 1

        # Predict using IDS system
        report = ids_system.predict(new_flow)

        # Update metrics
        total_flows = flow_idx
        detected_attacks = report["AlertMessage"].apply(lambda x: "ALERT" in x).sum()
        metrics_placeholder.metric("Total Flows Processed", total_flows)
        metrics_placeholder.metric("Detected Attacks", detected_attacks)

        # Update charts
        anomaly_counts.append(report.iloc[0]["AnomalyStatus"])
        attack_counts.append(report.iloc[0]["ThreatPrediction"])
        anomaly_chart_placeholder.bar_chart(pd.Series(anomaly_counts).value_counts())
        attack_chart_placeholder.bar_chart(pd.Series(attack_counts).value_counts())

        # Highlight alerts in table
        def highlight_alert(row):
            if "ALERT" in str(row["AlertMessage"]):
                return ["background-color: #ff9999"] * len(row)
            else:
                return [""] * len(row)

        table_placeholder.dataframe(report.style.apply(highlight_alert, axis=1))

        time.sleep(1)  # simulate real-time
else:
    st.warning("Please upload a CSV file to start simulation.")

# 🔹 STEP 19 — Save and Dowload Agentic Brain Files

In [ ]:
import os
import joblib
from google.colab import files

# 1. Create the folder
if not os.path.exists('models'):
    os.makedirs('models')

# 2. Save everything into that folder
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(anomaly_model, "models/anomaly_detector.pkl")
joblib.dump(threat_model, "models/threat_classifier.pkl")
joblib.dump(encoder, "models/label_encoder.pkl")

# 3. Zip the folder and download it immediately
!zip -r models.zip models/
files.download("models.zip")

print("✅ Success! Your browser should now be downloading 'models.zip'.")

# 🔹 STEP 20 — Install RAG Dependencies

In [ ]:
# Install LangChain and FAISS (Vector Database)
!pip install langchain langchain-community langchain-openai faiss-cpu pypdf
!pip install langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu pypdf


# 🔹 STEP 21 — Setup RAG Knowledge Base

1. Create a file named cyber_manual.txt in your Colab.

2. Paste this content into it (or upload a PDF of your choice):

    

  *   DDoS Mitigation: Increase bandwidth and use a scrubbing service.
  *   PortScan Mitigation: Close unused ports and configure firewalls.


  *   Brute Force Mitigation: Implement account lockout policies and MFA.






    

    

# 🔹 STEP 22 — Initialize the RAG Agent

In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-openai langchain-text-splitters
!pip install langchain==0.2.16 \
             langchain-community==0.2.16 \
             langchain-openai==0.1.23 \
             langchain-text-splitters==0.2.4 \
             faiss-cpu \
             pypdf

In [ ]:
!pip install -U langchain-text-splitters
!pip uninstall -y langchain langchain-core langchain-community langchain-openai

def setup_rag_system(file_path="cyber_manual.txt"):

    loader = TextLoader(file_path)
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    splits = splitter.split_documents(docs)

    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(splits, embeddings)
    retriever = vectorstore.as_retriever()

    llm = ChatOpenAI(model="gpt-4o-mini")

    prompt = ChatPromptTemplate.from_template(
        "Use the context to answer the question.\n\nContext:\n{context}\n\nQuestion:\n{question}"
    )

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs,
         "question": RunnablePassthrough()}
        | prompt
        | llm
    )

    return rag_chain

# 🔹 STEP 23 — Connect Agents to RAG (The "Orchestrator")

In [ ]:
def get_mitigation_advice(threat_type):
    query = f"How do I mitigate a {threat_type} attack?"
    return rag_agent.run(query)

# Example:
# threat = "DDoS"
# advice = get_mitigation_advice(threat)
# print(f"Agent Advice: {advice}")